# 04 · Retrieve — 08 The backend cascade

**Ported from `terrier-ta/course_rag.py`. `01-single-index-retrieval.ipynb` only ported the simple in-memory path -- this notebook adds the fallback pattern behind it: three interchangeable backends, tried in order, merged by taking the max score per chunk.**

Same shape as `02-chunk/03-store-backends.ipynb`'s write-side pattern (one
store chosen at runtime rather than a script per store) -- this is the read
side of that same idea. No Pinecone or pgvector credentials are needed:
both stubs behave exactly like `03-store-backends.ipynb`'s Pinecone
path -- they check for their own configuration and stop cleanly with a
named message when it's absent, so the cascade always finishes by falling
through to the local, in-memory store.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `search_pinecone_stub` | Returns `[]` when `PINECONE_API_KEY` isn't set, rather than raising | `search_pinecone_stub("bio201", "...", top_k=5)` |
| `search_pgvector_stub` | Same shape, for a pgvector backend that isn't wired up here | `search_pgvector_stub("bio201", "...", top_k=5)` |
| `search_local` | The one backend that actually runs -- an in-memory per-course store | `search_local("bio201", "...", top_k=5)` |
| `retrieve` | Tries each backend in order, merges results by max score per chunk id | `retrieve("bio201", "what do mitochondria do?")` |


In [ ]:
import sys
from pathlib import Path

_p = Path.cwd().resolve()
for _ in range(6):
    if (_p / "nbio.py").is_file():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")

import nbio

repo_root = nbio.bootstrap()
nbio.show_environment()

## Step 1 — a hash-embedded, in-memory course store

Same shape as `01-single-index-retrieval.ipynb`'s synthetic courses --
built fresh here so this notebook runs standalone. `hash_embed` is the
same deterministic, offline embedding from `03-embed/01-offline-embeddings.ipynb`:
real wiring, not semantically meaningful vectors, which is enough to prove
the cascade and merge logic works without needing any account anywhere.

In [ ]:
import hashlib
import math


def hash_embed(text: str, dim: int = 384) -> list[float]:
    vec = [0.0] * dim
    tokens = (text or "").lower().split()
    if not tokens:
        return vec
    for tok in tokens:
        h = int(hashlib.sha256(tok.encode("utf-8")).hexdigest(), 16)
        idx = h % dim
        sign = 1.0 if (h >> 8) & 1 else -1.0
        vec[idx] += sign
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]


def cosine(a: list[float], b: list[float]) -> float:
    if not a or not b or len(a) != len(b):
        return 0.0
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a)) or 1.0
    nb = math.sqrt(sum(x * x for x in b)) or 1.0
    return dot / (na * nb)


LOCAL_STORE: dict[str, list[dict]] = {
    "bio201": [
        {"chunk_id": "bio201::c0", "text": "Mitochondria convert nutrients into ATP through oxidative phosphorylation."},
        {"chunk_id": "bio201::c1", "text": "Photosynthesis converts light energy into chemical energy stored in glucose."},
    ],
}
for course in LOCAL_STORE.values():
    for ch in course:
        ch["embedding"] = hash_embed(ch["text"])

print(f"local store seeded: {list(LOCAL_STORE)} -- {len(LOCAL_STORE['bio201'])} chunks in bio201")

## Step 2 — the two stubs that stop cleanly, not with a stack trace

Ported behavior from `_search_pinecone` / `_search_pgvector`'s own
"not configured, return nothing" branch -- the real functions check
settings and return `[]` immediately when their backend isn't enabled,
they don't raise. That's what makes the cascade below safe to run with
nothing but the local store configured.

In [ ]:
import os


def search_pinecone_stub(course_id: str, query: str, top_k: int) -> list[dict]:
    if not os.environ.get("PINECONE_API_KEY"):
        return []
    raise NotImplementedError("a real Pinecone client would run here -- see 02-chunk/03-store-backends.ipynb")


def search_pgvector_stub(course_id: str, query: str, top_k: int) -> list[dict]:
    # No pgvector wiring in this cookbook at all -- always falls through.
    return []


print("pinecone stub, no key set:", search_pinecone_stub("bio201", "irrelevant", top_k=5))
print("pgvector stub, never wired here:", search_pgvector_stub("bio201", "irrelevant", top_k=5))

## Step 3 — `search_local`, the backend that actually runs

Ported from `_search_local`: embed the query, cosine-score every chunk in
that course's store, sort, take the top k. This is the only backend that
does real work in this notebook -- the point of the cascade is that
callers don't need to know that.

In [ ]:
def search_local(course_id: str, query: str, top_k: int) -> list[dict]:
    chunks = LOCAL_STORE.get(course_id, [])
    if not chunks:
        return []
    qvec = hash_embed(query)
    scored = [{**ch, "score": cosine(qvec, ch["embedding"]), "course_id": course_id} for ch in chunks]
    scored.sort(key=lambda x: -x["score"])
    return scored[:top_k]


for r in search_local("bio201", "what do mitochondria do?", top_k=2):
    print(f"  score={r['score']:.3f}  {r['text'][:60]!r}")

## Step 4 — `retrieve`: the cascade, merged by max score per chunk

Ported from `retrieve`'s `_ingest` pattern: try each backend in order,
and for every result seen, keep it only if its score beats whatever that
same `chunk_id` already has. With only the local backend actually
returning anything here, every result's source is `"local"` -- the
cascade order (pinecone -> pgvector -> local) is preserved so the pattern
is real even though only the last link ever fires in this repo.

In [ ]:
def retrieve(course_id: str, query: str, top_k: int = 5) -> list[dict]:
    merged: dict[str, dict] = {}

    def _ingest(docs: list[dict], source: str) -> None:
        for d in docs:
            key = d["chunk_id"]
            prev = merged.get(key)
            if prev is None or d["score"] > prev["score"]:
                merged[key] = {**d, "source": source}

    _ingest(search_pinecone_stub(course_id, query, top_k), "pinecone")
    if len(merged) < top_k:
        _ingest(search_pgvector_stub(course_id, query, top_k), "pgvector")
    if len(merged) < top_k:
        _ingest(search_local(course_id, query, top_k), "local")

    return sorted(merged.values(), key=lambda x: -x["score"])[:top_k]


results = retrieve("bio201", "what do mitochondria do?", top_k=2)
for r in results:
    print(f"  source={r['source']:<9} score={r['score']:.3f}  {r['text'][:55]!r}")

assert all(r["source"] == "local" for r in results), "with nothing else configured, everything should fall through to local"
print()
print("confirmed: cascade fell through pinecone -> pgvector -> local, exactly as configured")

## Where this differs from `01-single-index-retrieval.ipynb`

Notebook 01 assumes exactly one store and calls it directly. This notebook
adds the fallback: the *caller* of `retrieve()` never needs to know which
backend actually answered -- only that one did, recorded in `source` for
anyone who wants to check. Plugging in a real Pinecone or pgvector client
here is a matter of filling in `search_pinecone_stub` /
`search_pgvector_stub`'s bodies; the cascade and merge logic above does
not change.